# 00. 데이터 준비 (AI Hub 한국 이미지 - 음식)

AI Hub `한국 이미지(음식)` 데이터셋을 받아 224x224 컬러/그레이 한 쌍으로 정리합니다.
* 출처: https://www.aihub.or.kr/aihubdata/data/view.do?dataSetSn=79
* 구축 수량: 150종 × 1,000장 (총 ~150,000장, 15.73GB)
* 본 프로젝트는 train 12,922 / test 2,465 로 부분 샘플링하여 사용했습니다.

In [ ]:
import os, glob, zipfile
import numpy as np
from PIL import Image
from tqdm import tqdm

## 1. 경로 설정

Colab 사용 기준. 로컬에서는 `DATA_ROOT` 만 바꿔주면 됩니다.

In [ ]:
DATA_ROOT = '/content/drive/MyDrive/딥러닝/DATASET'  # AI Hub zip 파일들이 있는 위치
OUT_COLOR  = os.path.join(DATA_ROOT, 'color_image')
OUT_GRAY   = os.path.join(DATA_ROOT, 'gray_image')
os.makedirs(OUT_COLOR, exist_ok=True)
os.makedirs(OUT_GRAY,  exist_ok=True)

## 2. zip 추출 + 224x224 resize + gray 페어 생성

In [ ]:
def process_zip(zip_path, out_color, out_gray, size=(224, 224)):
    with zipfile.ZipFile(zip_path, 'r') as z:
        names = [n for n in z.namelist() if n.lower().endswith(('.jpg', '.jpeg', '.png'))]
        for name in tqdm(names, desc=os.path.basename(zip_path)):
            with z.open(name) as f:
                try:
                    img = Image.open(f).convert('RGB').resize(size)
                except Exception:
                    continue
                stem = os.path.splitext(os.path.basename(name))[0]
                img.save(os.path.join(out_color, stem + '_color.png'))
                img.convert('L').save(os.path.join(out_gray, stem + '_gray.png'))

for zp in sorted(glob.glob(os.path.join(DATA_ROOT, '*.zip'))):
    process_zip(zp, OUT_COLOR, OUT_GRAY)

## 3. train / test split

원 보고서와 동일하게 train 12,922 / test 2,465 로 분할합니다.

In [ ]:
import shutil, random
random.seed(42)
all_color = sorted(glob.glob(os.path.join(OUT_COLOR, '*.png')))
random.shuffle(all_color)
TRAIN_N, TEST_N = 12922, 2465
splits = {'train': all_color[:TRAIN_N], 'test': all_color[TRAIN_N:TRAIN_N+TEST_N]}
for split, files in splits.items():
    for sub in ('color_image', 'gray_image'):
        os.makedirs(os.path.join(DATA_ROOT, split, sub), exist_ok=True)
    for cf in tqdm(files, desc=split):
        gf = cf.replace('color_image', 'gray_image').replace('_color.png', '_gray.png')
        shutil.copy2(cf, os.path.join(DATA_ROOT, split, 'color_image', os.path.basename(cf)))
        if os.path.exists(gf):
            shutil.copy2(gf, os.path.join(DATA_ROOT, split, 'gray_image', os.path.basename(gf)))
print({s: len(fs) for s, fs in splits.items()})

## 4. 샘플 확인

In [ ]:
import matplotlib.pyplot as plt
sample_color = sorted(glob.glob(os.path.join(DATA_ROOT, 'train', 'color_image', '*.png')))[:5]
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for i, p in enumerate(sample_color):
    g = p.replace('color_image', 'gray_image').replace('_color.png', '_gray.png')
    axes[0, i].imshow(Image.open(p)); axes[0, i].axis('off'); axes[0, i].set_title('color')
    axes[1, i].imshow(Image.open(g), cmap='gray'); axes[1, i].axis('off'); axes[1, i].set_title('gray')
plt.tight_layout(); plt.show()